In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import os
import subprocess
import scgen
from scgen.file_utils import ensure_dir_for_file

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_versions()
sc.settings.set_figure_params(dpi=80)  # low dpi (dots per inch) yields small inline figures

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
2026-02-16 01:12:41.302041: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-16 01:12:41.402844: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH:

Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
train_path = "../data/pancreas.h5ad"
if os.path.isfile(train_path):
    adata = scgen.load_file(train_path)
else:
    train_url = "https://www.dropbox.com/s/qj1jlm9w10wmt0u/pancreas.h5ad?dl=1"
    t_dl = wget.download(train_url, train_path)
    adata = scgen.load_file(train_path)
adata = anndata.AnnData(X=np.expm1(adata.raw.X), var=adata.raw.var, obs=adata.obs)
sc.pp.filter_cells(adata, min_genes=600)

/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/sagemaker-user/.conda/envs/scgen-repro-env/lib/python3.10/site-packages/anndata/compat/__init__.py:371: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


filtered out 376 cells that have less than 600 genes expressed


Attempts to run Scanorama within a Jupyter Notebook have been unsuccessful. However, exporting the data outside of the notebook and running Scanorama's entire workflow, including turning the data into `.npz` pickles results in functional output. So let's export the rawest form of the count matrices for Scanorama to process.

This notebook won't run immediately on your system as you need a Scanorama GitHub clone.

In [3]:
df1 = pd.DataFrame(data=adata[adata.obs['sample']=='Baron'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Baron'].var_names,
                  columns=np.arange(np.sum(adata.obs['sample']=='Baron')))

df2 = pd.DataFrame(data=adata[adata.obs['sample']=='Muraro'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Muraro'].var_names,
                  columns=np.arange(np.sum(adata.obs['sample']=='Muraro')))

df3 = pd.DataFrame(data=adata[adata.obs['sample']=='Segerstolpe'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Segerstolpe'].var_names,
                  columns=np.arange(np.sum(adata.obs['sample']=='Segerstolpe')))

df4 = pd.DataFrame(data=adata[adata.obs['sample']=='Wang'].X.todense().transpose(),
                  index=adata[adata.obs['sample']=='Wang'].var_names,
                  columns=np.arange(np.sum(adata.obs['sample']=='Wang')))

df1.to_csv(ensure_dir_for_file('../../scanorama/data/4panc/baron.txt'),sep='\t')
df2.to_csv(ensure_dir_for_file('../../scanorama/data/4panc/muraro.txt'),sep='\t')
df3.to_csv(ensure_dir_for_file('../../scanorama/data/4panc/segerstolpe.txt'),sep='\t')
df4.to_csv(ensure_dir_for_file('../../scanorama/data/4panc/wang.txt'),sep='\t')

Run Scanorama. To do this, you need to create this file at `conf/4panc.txt`:

	data/4panc/baron
	data/4panc/muraro
	data/4panc/segerstolpe
	data/4panc/wang

...and this file at `bin/4panc.py`:

    import numpy as np
    from sklearn.preprocessing import normalize, LabelEncoder
    import sys

    from process import load_names, merge_datasets, save_datasets
    from scanorama import correct, visualize, process_data
    from scanorama import dimensionality_reduce

    import time
    from datetime import timedelta

    data_names = [
        'data/4panc/baron',
        'data/4panc/muraro',
        'data/4panc/segerstolpe',
        'data/4panc/wang'
    ]

    if __name__ == '__main__':
        datasets, genes_list, n_cells = load_names(data_names)
        t1 = time.time()
        datasets, genes = correct(datasets, genes_list)
        datasets = [ normalize(ds, axis=1) for ds in datasets ]
        t2 = time.time()
        print('Took '+str(timedelta(seconds=t2-t1)))

        save_datasets(datasets, genes, data_names)

As you can see, we have timing baked into the script, only measuring the time required to perform the actual batch correction. We export the resulting expression profiles so that we can analyse them within the notebook.

In [4]:
# Save notebook cwd so we can restore it (avoids FileNotFoundError when ../bbknn/examples/ does not exist)
notebook_cwd = os.getcwd()
os.chdir('../../scanorama/')

# So bin/process.py and bin/4panc.py can "import scanorama", add repo root to PYTHONPATH
scanorama_root = os.getcwd()
env = os.environ.copy()
env['PYTHONPATH'] = scanorama_root + os.pathsep + env.get('PYTHONPATH', '')

# Use sys.executable so the subprocess uses the same Python as the kernel (conda env with annoy, etc.)
import sys
subprocess.run([sys.executable, 'bin/process.py', 'conf/4panc.txt'], cwd=scanorama_root, env=env)
res = subprocess.run([sys.executable, 'bin/4panc.py'], cwd=scanorama_root, stdout=subprocess.PIPE, env=env)
print(res.stdout.decode('utf-8').split('\n'))

os.chdir(notebook_cwd)

Data names loaded
Successfully processed data/4panc/baron
Successfully processed data/4panc/muraro
Successfully processed data/4panc/segerstolpe
Successfully processed data/4panc/wang
['Loaded data/4panc/baron with 24516 genes and 8569 cells', 'Loaded data/4panc/muraro with 24516 genes and 2126 cells', 'Loaded data/4panc/segerstolpe with 24516 genes and 2987 cells', 'Loaded data/4panc/wang with 24516 genes and 635 cells', 'Found 14317 cells among all datasets', 'Found 24512 genes among all datasets', '[[0.         0.11194732 0.36692333 0.09291339]', ' [0.         0.         0.40263405 0.09606299]', ' [0.         0.         0.         0.84724409]', ' [0.         0.         0.         0.        ]]', 'Processing datasets (2, 3)', 'Processing datasets (1, 2)', 'Processing datasets (0, 2)', 'Processing datasets (0, 1)', 'Took 0:01:30.656122', '']


The last line in the standard output dump above captures a run time of two minutes. Now that that's done, import the expression back into the notebook and make a new object.

In [6]:
sc1 = pd.read_table('../../scanorama/data/4panc/baron.scanorama_corrected.txt',index_col=0)
sc2 = pd.read_table('../../scanorama/data/4panc/muraro.scanorama_corrected.txt',index_col=0)
sc3 = pd.read_table('../../scanorama/data/4panc/segerstolpe.scanorama_corrected.txt',index_col=0)
sc4 = pd.read_table('../../scanorama/data/4panc/wang.scanorama_corrected.txt',index_col=0)

adata_scanorama = anndata.AnnData(X=np.vstack((sc1.values.transpose(),sc2.values.transpose(),
                                               sc3.values.transpose(),sc4.values.transpose())),
                                  obs=adata.obs, var=pd.DataFrame(index=sc1.index))

The final object export.

In [7]:
adata_scanorama.write(ensure_dir_for_file('objects-pancreas/pancreas_scanorama.h5ad'))